## GRIDSEARCHCV

In [5]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.utils import shuffle
import os

# Limit threading for stability
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

# Load MNIST dataset
mnist = fetch_openml("mnist_784", version=1)
X, y = mnist['data'], mnist['target']

# Convert data to numeric types
X = X.to_numpy().astype('float32')
y = y.to_numpy().astype('int')

# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split dataset
X_train, X_test, y_train, y_test = X[:60000], X[60000:], y[:60000], y[60000:]

# Apply PCA to reduce dimensionality
pca = PCA(n_components=50, random_state=42)
X_train_reduced = pca.fit_transform(X_train)
X_test_reduced = pca.transform(X_test)

# Debug: Test a single KNN model
knn = KNeighborsClassifier(n_neighbors=3, weights="uniform", algorithm='kd_tree')
knn.fit(X_train_reduced, y_train)
y_pred = knn.predict(X_test_reduced)
print("Single KNN Model Test Accuracy:", accuracy_score(y_test, y_pred))

# Subsample data for GridSearchCV
X_train_small, y_train_small = shuffle(X_train_reduced, y_train, n_samples=5000, random_state=42)

# Define parameter grid for GridSearchCV
param_grid = {
    "n_neighbors": [3, 5],
    "weights": ["uniform", "distance"],
    "algorithm": ["kd_tree", "ball_tree"]
}

grid_search = GridSearchCV(
    estimator=KNeighborsClassifier(),
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    error_score='raise',
    return_train_score=True,
    n_jobs=1
)

# Fit GridSearchCV
grid_search.fit(X_train_small, y_train_small)
print("Best Parameters:", grid_search.best_params_)
print("Best Cross-Validated Score:", grid_search.best_score_)

# Train the best KNN model on the full dataset
best_knn = grid_search.best_estimator_
best_knn.fit(X_train_reduced, y_train)
y_pred_best = best_knn.predict(X_test_reduced)
print("Best KNN Model Test Accuracy:", accuracy_score(y_test, y_pred_best))

# Use RandomForestClassifier as a backup
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42)
rf_clf.fit(X_train_reduced, y_train)
y_pred_rf = rf_clf.predict(X_test_reduced)
print("Random Forest Model Test Accuracy:", accuracy_score(y_test, y_pred_rf))


d:\Anaconda\Lib\site-packages\sklearn\datasets\_openml.py:968: FutureWarning: The default value of `parser` will change from `'liac-arff'` to `'auto'` in 1.4. You can set `parser='auto'` to silence this warning. Therefore, an `ImportError` will be raised from 1.4 if the dataset is dense and pandas is not installed. Note that the pandas parser may return different data types. See the Notes Section in fetch_openml's API doc for details.
  warn(


Single KNN Model Test Accuracy: 0.9575
Best Parameters: {'algorithm': 'kd_tree', 'n_neighbors': 3, 'weights': 'distance'}
Best Cross-Validated Score: 0.922200073790764
Best KNN Model Test Accuracy: 0.9583
Random Forest Model Test Accuracy: 0.944


## RANDOMSEARCHCV

In [15]:
# Import necessary libraries
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score, RandomizedSearchCV
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score
import scipy as sp
import numpy as np

# Example dataset (replace with your actual data)
X = [
    "This is a sample text.",
    "Another example text data.",
    "Naive Bayes classifier works on text.",
    "CountVectorizer transforms text to features.",
    "RandomizedSearchCV tunes the hyperparameters."
]  # Replace with your dataset
y = [0, 1, 0, 1, 0]  # Replace with your actual labels

# Step 1: Convert text to numerical data
vectorizer = CountVectorizer()
X_vectorized = vectorizer.fit_transform(X).toarray()

# Step 2: Handle class imbalance using SMOTE with adjusted k_neighbors
smote = SMOTE(random_state=42, k_neighbors=1)  # Adjust k_neighbors to avoid errors
X_resampled, y_resampled = smote.fit_resample(X_vectorized, y)

# Step 3: Create a pipeline
pipe = Pipeline([
    ('vectorizer', CountVectorizer()),  # Text preprocessing
    ('classifier', MultinomialNB())    # Classification
])

# Step 4: Perform cross-validation with fewer splits
cv = StratifiedKFold(n_splits=3)  # Use fewer splits for small datasets
cv_score = cross_val_score(pipe, X, y, cv=cv, scoring='accuracy').mean()
print("Cross-validated Accuracy:", cv_score)

# Step 5: Define parameters for RandomizedSearchCV
params = {
    'vectorizer__min_df': [0, 1],  # Adjust to avoid feature filtering issues
    'vectorizer__lowercase': [True, False],
    'classifier__alpha': sp.stats.uniform(scale=1),  # Uniform distribution for alpha
}

# Step 6: Perform RandomizedSearchCV
rand = RandomizedSearchCV(
    estimator=pipe,
    param_distributions=params,
    n_iter=10,
    cv=cv,  # Use StratifiedKFold
    scoring='accuracy',
    error_score=np.nan,  # Ignore errors
    random_state=1
)
rand.fit(X, y)

# Step 7: Display results
print("Best Cross-validated Score:", rand.best_score_)
print("Best Parameters:", rand.best_params_)


Cross-validated Accuracy: 0.6666666666666666
Best Cross-validated Score: 0.6666666666666666
Best Parameters: {'classifier__alpha': 0.417022004702574, 'vectorizer__lowercase': True, 'vectorizer__min_df': 0}


d:\Anaconda\Lib\site-packages\sklearn\model_selection\_split.py:700: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(
d:\Anaconda\Lib\site-packages\sklearn\model_selection\_split.py:700: UserWarning: The least populated class in y has only 2 members, which is less than n_splits=3.
  warnings.warn(
